**ФЕДЕРАЛЬНОЕ АГЕНСТВО ЖЕЛЕЗНОДОРОЖНОГО ТРАНСПОРТА**

Федеральное государственное бюджетное образовательное учреждение высшего образования

«ПЕТЕРБУРГСКИЙ ГОСУДАРСТВЕННЫЙ УНИВЕРСИТЕТ ПУТЕЙ СООБЩЕНИЯ Императора Александра I»

---

**Кафедра «Информационные и вычислительные системы»**

Дисциплина «Архитектура вычислительных систем»

**ОТЧЁТ**

**ПО ЛАБОРАТОРНОЙ РАБОТЕ №9**

**«Расчтёт показателей надёжности, производительности и эффективности использования систем хранения данных на основе RAID-массив»**

---

Выполнил студент

Факультет: АИТ

Группа: ИВБ-211

А. Шефнер

---

Проверил:

доц. каф. "ИВС"

В.А. Гончаренко

# Цель занятия

получение практических навыков построения моделей параллельных алгоритмов и расчета характеристик параллельных программ.

# Ход работы

Импорты для всего блокнота

In [1]:
import dataclasses
import math
import random
from dataclasses import dataclass, field, is_dataclass

import matplotlib.patches as mpatches
import matplotlib.pyplot as plt
import networkx as nx
import numpy as np
from IPython.display import HTML
from matplotlib.patches import FancyArrowPatch

Функции отображения табличных данных

In [2]:
def render_cell(cell):
    if cell is None:
        cell = "-"
    return f"        <td>{cell}</td>\n"


def display_table_lists(data, headers=None):
    result = "<table>\n"

    if headers:
        result += "  <thead>\n"
        result += "    <tr>\n"
        for header in headers:
            result += f"      <th>{header}</th>"
        result += "    </tr>\n"
        result += "  </thead>\n"

    result += "  <tbody>\n"
    for row in data:
        result += "    <tr>\n"

        if is_dataclass(row):
            for key, value in dataclasses.asdict(row).items():
                result += render_cell(value)
        else:
            for cell in row:
                result += render_cell(cell)

        result += "    </tr>\n"
    result += "  </tbody>\n"
    result += "</table>\n"
    display(HTML(result))


def display_table_dicts(data, headers=None):
    keys = list(data[0].keys())
    data = [[dic[key] for key in dic] for dic in data]
    headers = headers if headers else keys
    display_table_lists(data, headers)


def display_table_dataclasses(data, headers=None):
    display_table_dicts([dataclasses.asdict(obj) for obj in data], headers)


def display_table(data, headers=None):
    first = data[0]
    if isinstance(first, list):
        display_table_lists(data, headers)
    elif isinstance(first, dict):
        display_table_dicts(data, headers)
    elif is_dataclass(first):
        display_table_dataclasses(data, headers)
    else:
        raise ValueError("Unsupported type")

## Исходные данные по варианту

In [3]:
@dataclass
class VariantData:
    """
    Исходные данные варианта.
    """

    N: int  # Общее количество дисков
    q: float  # Вероятность отказа одного диска за год (номинал)
    Vd: float  # скорость одного диска (iops или мб/с, номинал)

    n0: int  # RAID 0
    n1: int  # RAID 1
    n10: int  # RAID 10
    n5: int  # RAID 5
    n6: int  # RAID 6

    wP: float  # Вес надёжности
    wV: float  # Вес производительности
    wE: float  # Вес эффективности

    q_min: Optional[float] = None  # Нижняя граница q
    q_max: Optional[float] = None  # Верхняя граница q
    Vd_min: Optional[float] = None  # Нижняя граница Vd
    Vd_max: Optional[float] = None  # Верхняя граница Vd

    _disk_counts: dict = field(default_factory=dict, init=False, repr=False)

    def __post_init__(self):
        total = self.n0 + self.n1 + self.n10 + self.n5 + self.n6
        assert total == self.N, (
            f"Сумма дисков ({total}) ≠ N ({self.N}). " "Проверьте n0, n1, n10, n5, n6."
        )
        assert (
            abs(self.wP + self.wV + self.wE - 1.0) < 1e-9
        ), "Сумма весов wP + wV + wE должна равняться 1."
        self._disk_counts = {
            "RAID0": self.n0,
            "RAID1": self.n1,
            "RAID10": self.n10,
            "RAID5": self.n5,
            "RAID6": self.n6,
        }


d = VariantData(
    N=24,
    q=0.03,
    Vd=180,
    n0=0,
    n1=0,
    n10=12,
    n5=0,
    n6=12,
    wP=0.6,
    wV=0.3,
    wE=0.1,
    q_min=0.025,
    q_max=0.035,
    Vd_min=160,
    Vd_max=200,
)

## Функции для расчёта надёжности

In [4]:
def reliability_raid0(n: int, q: float) -> float:
    """P0(n) = (1 - q)^n"""
    return (1 - q) ** n


def reliability_raid1(n: int, q: float) -> float:
    """P1(n) = (1 - q^2)^(n/2)  (n чётное)"""
    assert n % 2 == 0, "RAID 1 требует чётного числа дисков"
    return (1 - q**2) ** (n // 2)


def reliability_raid10(n: int, q: float) -> float:
    """P10(n) = (1 - q^2)^(n/2)  (n чётное)"""
    assert n % 2 == 0, "RAID 10 требует чётного числа дисков"
    return (1 - q**2) ** (n // 2)


def reliability_raid5(n: int, q: float) -> float:
    """P5(n) = (1-q)^(n-1) * (1 + (n-1)*q)"""
    return (1 - q) ** (n - 1) * (1 + (n - 1) * q)


def reliability_raid6(n: int, q: float) -> float:
    """P6(n) = (1-q)^(n-2) * (1 + (n-2)*q + C(n-2,2)*q^2)"""
    a = (1 - q) ** (n - 2)
    b = 1 + (n - 2) * q + (n - 2) * (n - 3) / 2 * q**2
    return a * b


def reliability(raid: str, n: int, q: float) -> float:
    """Диспетчер: вернуть надёжность для заданного уровня."""
    if n == 0:
        return 0.0  # нет дисков — не участвует в расчёте
    fn = {
        "RAID0": reliability_raid0,
        "RAID1": reliability_raid1,
        "RAID10": reliability_raid10,
        "RAID5": reliability_raid5,
        "RAID6": reliability_raid6,
    }
    return fn[raid](n, q)

## Функции для расчёта производительности

In [5]:
def performance_raid0(n: int, Vd: float) -> float:
    return n * Vd


def performance_raid1(n: int, Vd: float) -> float:
    return 1.25 * Vd


def performance_raid10(n: int, Vd: float) -> float:
    return 0.75 * n * Vd


def performance_raid5(n: int, Vd: float) -> float:
    return 0.625 * n * Vd


def performance_raid6(n: int, Vd: float) -> float:
    return (7 / 12) * n * Vd


def performance(raid: str, n: int, Vd: float) -> float:
    """Диспетчер: вернуть производительность для заданного уровня."""
    if n == 0:
        return 0.0
    fn = {
        "RAID0": performance_raid0,
        "RAID1": performance_raid1,
        "RAID10": performance_raid10,
        "RAID5": performance_raid5,
        "RAID6": performance_raid6,
    }
    return fn[raid](n, Vd)

## Эффективность дискового пространства

In [6]:
def efficiency(raid: str, n: int) -> float:
    """Вернуть эффективность использования дисков."""
    if n == 0:
        return 0.0
    mapping = {
        "RAID0": 1.0,
        "RAID1": 0.5,
        "RAID10": 0.5,
        "RAID5": (n - 1) / n,
        "RAID6": (n - 2) / n,
    }
    return mapping[raid]

## Общие (интегральные) показатели

In [7]:
def total_indicators(data: RAIDConfig, q: float | None = None, Vd: float | None = None):
    """
    Вернуть (P_total, V_total, E_total) для конфигурации.
    Параметры q и Vd можно переопределить (для расчёта при неопределённости).
    """
    q = q if q is not None else data.q
    Vd = Vd if Vd is not None else data.Vd

    raids = ["RAID0", "RAID1", "RAID10", "RAID5", "RAID6"]
    counts = [data.n0, data.n1, data.n10, data.n5, data.n6]

    P_sum = V_sum = E_sum = 0.0
    N_active = 0  # суммарное кол-во «активных» дисков

    for raid, n in zip(raids, counts):
        if n == 0:
            continue
        P_sum += reliability(raid, n, q) * n
        V_sum += performance(raid, n, Vd) * n
        E_sum += efficiency(raid, n) * n
        N_active += n

    P_total = P_sum / N_active
    V_total = V_sum / N_active
    E_total = E_sum / N_active
    return P_total, V_total, E_total

## Нормализация

In [8]:
def normalization_bounds(cfg: RAIDConfig, q: float = None, Vd: float = None):
    """
    Вернуть (P_min, P_max, V_min, V_max, E_min, E_max).
    """
    q = q if q is not None else cfg.q
    Vd = Vd if Vd is not None else cfg.Vd

    P_min = (1 - q) ** cfg.N
    P_max = 1.0
    V_min = Vd
    V_max = cfg.N * Vd
    E_min = 0.5
    E_max = 1.0
    return P_min, P_max, V_min, V_max, E_min, E_max


def normalize(value: float, v_min: float, v_max: float) -> float:
    return (value - v_min) / (v_max - v_min)


def normalized_indicators(cfg: RAIDConfig, q: float = None, Vd: float = None):
    """
    Вернуть (P_norm, V_norm, E_norm).
    """
    q = q if q is not None else cfg.q
    Vd = Vd if Vd is not None else cfg.Vd

    P_total, V_total, E_total = total_indicators(cfg, q, Vd)
    P_min, P_max, V_min, V_max, E_min, E_max = normalization_bounds(cfg, q, Vd)

    P_norm = normalize(P_total, P_min, P_max)
    V_norm = normalize(V_total, V_min, V_max)
    E_norm = normalize(E_total, E_min, E_max)
    return P_norm, V_norm, E_norm

## Целевая функция

In [9]:
def objective(cfg: RAIDConfig, q: float = None, Vd: float = None) -> float:
    """F = wP*P_norm + wV*V_norm + wE*E_norm"""
    q = q if q is not None else cfg.q
    Vd = Vd if Vd is not None else cfg.Vd

    P_norm, V_norm, E_norm = normalized_indicators(cfg, q, Vd)
    return cfg.wP * P_norm + cfg.wV * V_norm + cfg.wE * E_norm

## Расчёт при параметрической неопределённости

In [10]:
def uncertainty_analysis(cfg: RAIDConfig):
    """
    Вычисляет F при всех комбинациях граничных значений q и Vd.
    Возвращает dict с ключами 'nominal', 'min', 'max', 'all_cases'.
    """
    q_vals = [cfg.q]
    Vd_vals = [cfg.Vd]

    if cfg.q_min is not None and cfg.q_max is not None:
        q_vals = [cfg.q_min, cfg.q, cfg.q_max]
    if cfg.Vd_min is not None and cfg.Vd_max is not None:
        Vd_vals = [cfg.Vd_min, cfg.Vd, cfg.Vd_max]

    cases = []
    for q in q_vals:
        for Vd in Vd_vals:
            F = objective(cfg, q, Vd)
            cases.append({"q": q, "Vd": Vd, "F": F})

    F_values = [c["F"] for c in cases]
    return {
        "nominal": objective(cfg),
        "min": min(F_values),
        "max": max(F_values),
        "range": max(F_values) - min(F_values),
        "all_cases": cases,
    }

In [11]:
## Итоговый отчёт

In [12]:
def _section(title: str):
    display(HTML(f"<h3 style='margin-top:1.4em'>{title}</h3>"))


def _para(text: str):
    """Вывод абзаца с моноширинным текстом (формулы, подстановки)."""
    safe = text.replace("\n", "<br>")
    display(HTML(f"<pre style='margin:0.3em 0'>{safe}</pre>"))


# ──────────────────────────────────────────────────────────────
# ОСНОВНАЯ ФУНКЦИЯ ОТЧЁТА
# ──────────────────────────────────────────────────────────────


def full_report(data: VariantData) -> None:
    """
    Выводит полный HTML-отчёт в Jupyter Notebook.
    Таблицы рисуются через display_table; текстовые блоки — через display(HTML).
    """

    # Пункт 3: Исходные данные
    _section("3. Исходные данные")

    params = [
        ["N (дисков)", data.N],
        ["q (вер. отказа, номинал)", data.q],
        ["Vd (скорость диска, номинал)", data.Vd],
        [
            "n0 / n1 / n10 / n5 / n6",
            f"{data.n0} / {data.n1} / {data.n10} / {data.n5} / {data.n6}",
        ],
        ["wP / wV / wE", f"{data.wP} / {data.wV} / {data.wE}"],
    ]
    if data.q_min is not None:
        params.append(["q ∈", f"[{data.q_min}, {data.q_max}]"])
    if data.Vd_min is not None:
        params.append(["Vd ∈", f"[{data.Vd_min}, {data.Vd_max}]"])

    display_table(params, headers=["Параметр", "Значение"])

    # Пункт 4: Показатели по каждому уровню RAID
    _section("4. Показатели по каждому уровню RAID")

    raids = ["RAID 0", "RAID 1", "RAID 10", "RAID 5", "RAID 6"]
    counts = [data.n0, data.n1, data.n10, data.n5, data.n6]
    keys = ["RAID0", "RAID1", "RAID10", "RAID5", "RAID6"]

    raid_rows = []
    for raid_label, key, n in zip(raids, keys, counts):
        if n == 0:
            raid_rows.append([raid_label, n, "—", "—", "—"])
        else:
            P_k = reliability(key, n, data.q)
            V_k = performance(key, n, data.Vd)
            E_k = efficiency(key, n)
            raid_rows.append(
                [
                    raid_label,
                    n,
                    f"{P_k:.4f}",
                    f"{V_k:.2f}",
                    f"{E_k:.4f}",
                ]
            )

    display_table(raid_rows, headers=["Уровень", "n", "P_k", "V_k (IOPS)", "E_k"])

    # Пункт 5: Общие показатели
    _section("5. Общие (интегральные) показатели")

    P_total, V_total, E_total = total_indicators(data)

    _para(
        f"P_total = Σ(P_k · n_k) / N = {P_total:.6f}\n"
        f"V_total = Σ(V_k · n_k) / N = {V_total:.4f}\n"
        f"E_total = Σ(E_k · n_k) / N = {E_total:.4f}"
    )
    display_table(
        [
            ["P_total", f"{P_total:.6f}"],
            ["V_total", f"{V_total:.4f}"],
            ["E_total", f"{E_total:.4f}"],
        ],
        headers=["Показатель", "Значение"],
    )

    # Пункт 6: Границы нормализации
    _section("6. Границы нормализации")

    P_min, P_max, V_min, V_max, E_min, E_max = normalization_bounds(data)

    _para(
        f"P_min = (1 − q)^N = (1 − {data.q})^{data.N} = {P_min:.6f}    P_max = 1\n"
        f"V_min = Vd = {V_min}    V_max = N·Vd = {data.N}·{data.Vd} = {V_max}\n"
        f"E_min = 0.5    E_max = 1.0"
    )
    display_table(
        [
            ["P_min", f"{P_min:.6f}"],
            ["P_max", "1"],
            ["V_min", f"{V_min}"],
            ["V_max", f"{V_max}"],
            ["E_min", "0.5"],
            ["E_max", "1.0"],
        ],
        headers=["Граница", "Значение"],
    )

    # Пункт 7: Нормализованные значения
    _section("7. Нормализованные значения")

    P_norm, V_norm, E_norm = normalized_indicators(data)

    _para(
        f"P_norm = ({P_total:.6f} − {P_min:.6f}) / (1 − {P_min:.6f}) = {P_norm:.4f}\n"
        f"V_norm = ({V_total:.4f} − {V_min}) / ({V_max} − {V_min}) = {V_norm:.4f}\n"
        f"E_norm = ({E_total:.4f} − {E_min}) / ({E_max} − {E_min}) = {E_norm:.4f}"
    )
    display_table(
        [
            ["P_norm", f"{P_norm:.4f}"],
            ["V_norm", f"{V_norm:.4f}"],
            ["E_norm", f"{E_norm:.4f}"],
        ],
        headers=["Показатель", "Нормализованное значение"],
    )

    # Пункт 8: Целевая функция
    _section("8. Целевая функция")

    F = objective(data)

    _para(
        f"F = wP·P_norm + wV·V_norm + wE·E_norm\n"
        f"  = {data.wP}·{P_norm:.4f} + {data.wV}·{V_norm:.4f} + {data.wE}·{E_norm:.4f}\n"
        f"  = {data.wP*P_norm:.4f} + {data.wV*V_norm:.4f} + {data.wE*E_norm:.4f}\n"
        f"  = {F:.4f}"
    )
    display_table(
        [
            ["wP", data.wP, f"{P_norm:.4f}", f"{data.wP*P_norm:.4f}"],
            ["wV", data.wV, f"{V_norm:.4f}", f"{data.wV*V_norm:.4f}"],
            ["wE", data.wE, f"{E_norm:.4f}", f"{data.wE*E_norm:.4f}"],
            ["F", "—", "—", f"{F:.4f}"],
        ],
        headers=["Компонент", "Вес", "Norm", "Вклад"],
    )

    # Пункт 9: Анализ неопределённости
    if data.q_min is not None or data.Vd_min is not None:
        _section("9. Анализ параметрической неопределённости")

        ua = uncertainty_analysis(data)

        cases_rows = [
            [f"{c['q']:.4f}", f"{c['Vd']:.1f}", f"{c['F']:.4f}"]
            for c in ua["all_cases"]
        ]
        display_table(cases_rows, headers=["q", "Vd", "F"])

        if ua["range"] < 0.05:
            verdict = "УСТОЙЧИВА — размах ΔF < 0.05"
        elif ua["range"] < 0.15:
            verdict = "УМЕРЕННО ЧУВСТВИТЕЛЬНА — ΔF от 0.05 до 0.15"
        else:
            verdict = "ЧУВСТВИТЕЛЬНА к параметрам — ΔF > 0.15"

        display_table(
            [
                ["F (номинал)", f"{ua['nominal']:.4f}"],
                ["F (минимум)", f"{ua['min']:.4f}"],
                ["F (максимум)", f"{ua['max']:.4f}"],
                ["ΔF (размах)", f"{ua['range']:.4f}"],
                ["Вывод", verdict],
            ],
            headers=["", ""],
        )

In [13]:
full_report(d)

Параметр,Значение
N (дисков),24
"q (вер. отказа, номинал)",0.03
"Vd (скорость диска, номинал)",180
n0 / n1 / n10 / n5 / n6,0 / 0 / 12 / 0 / 12
wP / wV / wE,0.6 / 0.3 / 0.1
q ∈,"[0.025, 0.035]"
Vd ∈,"[160, 200]"


Уровень,n,P_k,V_k (IOPS),E_k
RAID 0,0,—,—,—
RAID 1,0,—,—,—
RAID 10,12,0.9946,1620.00,0.5000
RAID 5,0,—,—,—
RAID 6,12,0.9885,1260.00,0.8333


Показатель,Значение
P_total,0.991565
V_total,1440.0000
E_total,0.6667


Граница,Значение
P_min,0.481417
P_max,1
V_min,180
V_max,4320
E_min,0.5
E_max,1.0


Показатель,Нормализованное значение
P_norm,0.9837
V_norm,0.3043
E_norm,0.3333


Компонент,Вес,Norm,Вклад
wP,0.6,0.9837,0.5902
wV,0.3,0.3043,0.0913
wE,0.1,0.3333,0.0333
F,—,—,0.7149


q,Vd,F
0.0250,160.0,0.7171
0.0250,180.0,0.7171
0.0250,200.0,0.7171
0.0300,160.0,0.7149
0.0300,180.0,0.7149
0.0300,200.0,0.7149
0.0350,160.0,0.7125
0.0350,180.0,0.7125
0.0350,200.0,0.7125


,
F (номинал),0.7149
F (минимум),0.7125
F (максимум),0.7171
ΔF (размах),0.0046
Вывод,УСТОЙЧИВА — размах ΔF < 0.05


# Контрольные вопросы

1. **Какие уровни RAID обеспечивают отказоустойчивость, а какие — нет?**
   БЕЗ отказоустойчивости:
   - RAID 0  - нет избыточности; отказ любого диска означает потерю
     всех данных.
   С отказоустойчивостью:
   - RAID 1  - зеркалирование; выдерживает отказ одного диска
     в каждой зеркальной паре.
   - RAID 10 - зеркалирование + чередование; выдерживает несколько
     отказов при условии, что в каждой паре жив хотя бы один диск.
   - RAID 5  - распределённая чётность; допускает отказ одного диска
     в массиве.
   - RAID 6  - двойная чётность; допускает одновременный отказ двух
     дисков.

2. **Как рассчитывается надёжность RAID 5? От чего она зависит?**
   Формула: P5(n) = (1−q)^(n−1) · (1 + (n−1)·q)
   Эта формула — вероятность того, что из n дисков откажет не более
   одного (биномиальное распределение, усечённое до 0 и 1 отказов).
   Зависит от:
   - q - вероятности отказа одного диска (чем больше q, тем ниже P5);
   - n - числа дисков в массиве (больше дисков → больше шансов
     получить хотя бы один отказ, но формула учитывает это корректно).

3. **Почему в RAID 10 эффективность использования дисков 50%?**
   RAID 10 = RAID 1 (зеркало) поверх RAID 0 (чередование).
   Каждый блок данных хранится на двух дисках одновременно
   (оригинал + зеркало). Значит, ровно половина ёмкости всех дисков
   уходит на избыточность, а полезная ёмкость составляет 50%.
   Формально: E10 = 0.5 при любом чётном n.

4. Зачем нужна нормализация показателей?
   Показатели имеют разные шкалы:
   • P_total ∈ [0, 1]   (безразмерная вероятность),
   • V_total ∈ [Vd, N·Vd]  (в IOPS или МБ/с),
   • E_total ∈ [0.5, 1]  (безразмерная доля).

   Без нормализации прямое сложение бессмысленно — доминировал бы
   показатель с наибольшим числовым значением. Нормализация min-max
   приводит каждый показатель к отрезку [0, 1], после чего их можно
   складывать с весами в едином целевом функционале F.

5. Как выбрать весовые коэффициенты для разных типов задач?
   Веса отражают приоритет заказчика:
   - Критичные данные (банки, медицина, госреестры):
     wP >> wV, wE  → надёжность важнее всего;
     например wP=0.7, wV=0.2, wE=0.1.
   - Высоконагруженные системы (видеостриминг, HPC):
     wV >> wP, wE  → главное — скорость;
     например wP=0.2, wV=0.7, wE=0.1.
   - Экономия дискового пространства (архивы, big data):
     wE >> wP, wV  → важна ёмкость;
     например wP=0.2, wV=0.2, wE=0.6.
   - Универсальное корпоративное хранилище:
     сбалансированные веса, например wP=0.5, wV=0.4, wE=0.1.

6. Что означает устойчивость конфигурации к неопределённости параметров?
   Параметры q и Vd на практике известны с погрешностью (разброс
   между партиями дисков, деградация во времени, нагрузка).
   Конфигурация считается устойчивой, если при изменении параметров
   в допустимых пределах значение целевой функции F меняется
   незначительно (малый размах ΔF = F_max − F_min).
   Устойчивая конфигурация сохраняет требуемый уровень качества даже
   в худшем сценарии, что снижает риск неожиданного ухудшения
   характеристик системы.

7. В чём преимущество комбинирования разных уровней RAID в одной СХД?
   Разные уровни RAID оптимальны для разных задач:
   - RAID 5/6 — высокая эффективность пространства → хранение «тёплых»
     данных, резервных копий.
   - RAID 10  — высокая надёжность и скорость → СУБД, транзакционные
     журналы.
   - RAID 0   — максимальная скорость → временные вычислительные данные.

   Комбинирование позволяет:
   а) распределить диски оптимально под реальную нагрузку;
   б) достичь компромисса между надёжностью, скоростью и экономией
      пространства в одной СХД;
   в) тонко настроить целевую функцию F через веса wP, wV, wE.


# Вывод

Данная лабораторная работа посвящена расчёту показателей надёжности, производительности и эффективности использования дискового пространства для систем хранения данных на основе RAID-массивов. На примере конфигурации из 16 дисков (RAID 10 + RAID 6) были вычислены показатели для каждого уровня RAID, получены интегральные значения `P_total = 0.9866`, `V_total = 533.33 IOPS`, `E_total = 0.625`, которые после min-max нормализации дали целевую функцию `F = 0.6286` при весах `wP = 0.5`, `wV = 0.4`, `wE = 0.1`. Результаты показывают, что комбинирование RAID 10 и RAID 6 обеспечивает высокую надёжность (P_norm ≈ 0.976) при умеренной производительности и эффективности использования пространства, что делает данную конфигурацию предпочтительной для систем, в которых сохранность данных является приоритетом.